# Projet 14 sur Kaggle

Ce notebook ne change pas d'une étape à l'autre : **il n'est importé qu'une fois**. Le travail
de chaque étape est dans un script du dépôt, `scripts/kaggle/<étape>.sh`, que la cellule 2
récupère à jour à chaque lancement.

Utilisation : *Run All*, après avoir choisi l'étape dans la dernière cellule.

Prérequis, réglés une fois dans le panneau de droite :

- **Accelerator** : GPU T4 x2 ;
- **Internet** : activé ;
- **Secrets** : `HF_TOKEN`, `WANDB_API_KEY` et `GITHUB_TOKEN` (jeton GitHub *fine-grained*,
  lecture seule sur le contenu du dépôt `finetunez-votre-propre-llm`), cochés pour ce notebook.

Le code est cloné dans `/tmp/projet14`, hors de `/kaggle/working` : seul ce dernier est
sauvegardé avec le notebook, et les étapes n'y copient que leurs résultats.

## 1. Secrets

In [ ]:
import os

NAMES = ("HF_TOKEN", "WANDB_API_KEY", "GITHUB_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient  # Kaggle

    secrets = UserSecretsClient()
    for name in NAMES:
        os.environ[name] = secrets.get_secret(name)
except ImportError:
    from google.colab import userdata  # Colab, en secours

    for name in NAMES:
        os.environ[name] = userdata.get(name)

os.environ["WANDB_PROJECT"] = "triage-chsa"
print({name: "ok" if os.environ.get(name) else "MANQUANT" for name in NAMES})

## 2. Code du projet (toujours la dernière version de la branche)

In [ ]:
import base64
import subprocess
from pathlib import Path

REPO = "https://github.com/Formation-AI-Engineer/finetunez-votre-propre-llm.git"
BRANCH = "dev"
WORKDIR = Path("/tmp/projet14")

# Le jeton passe par un en-tête HTTP : il n'est écrit ni dans l'URL ni dans .git/config.
token = base64.b64encode(f"x-access-token:{os.environ['GITHUB_TOKEN']}".encode()).decode()
auth = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {token}"]


def git(*args, cwd=None):
    subprocess.run(["git", *auth, *args], cwd=cwd, check=True)


if (WORKDIR / ".git").exists():
    git("fetch", "--quiet", "origin", BRANCH, cwd=WORKDIR)
    git("checkout", "--quiet", BRANCH, cwd=WORKDIR)
    git("reset", "--quiet", "--hard", f"origin/{BRANCH}", cwd=WORKDIR)
else:
    git("clone", "--quiet", "--branch", BRANCH, REPO, str(WORKDIR))
os.chdir(WORKDIR)
!git log --oneline -1

## 3. Environnement : Python 3.12, dépendances de `uv.lock`, contrôle du GPU

Environ 2 minutes au premier lancement d'une session, quelques secondes ensuite.

In [ ]:
!bash scripts/kaggle/setup.sh

## 4. Étape à lancer

Changez seulement `STEP` : c'est le nom d'un script de `scripts/kaggle/` (la liste s'affiche).

In [ ]:
STEP = "baseline"

!ls scripts/kaggle
!bash scripts/kaggle/{STEP}.sh